# Teleport + Amazon AgentCore Gateway: Identity-Aware AI Tooling
## Verified caller identity from Teleport flows through every AI tool call

## Overview

This demo shows how Teleport and Amazon Bedrock AgentCore Gateway work together to bring
zero-trust identity to AI tool access. The same identity that controls infrastructure access
in Teleport now controls AI tool access — verified, auditable, and policy-driven at every hop.

**The key insight:** Teleport issues a signed JWT when a developer runs `tsh mcp connect`.
AgentCore Gateway validates that JWT against Teleport's OIDC endpoint before any tool runs.
A REQUEST interceptor Lambda then decodes the validated JWT and injects the caller's identity
into the tool invocation — so every Lambda tool knows *who* called it.

### Architecture

```
Developer
  │  tsh login → tsh mcp connect agentcore-gateway
  ↓
Teleport Proxy  (issues JWT: sub, roles)
  │  mcp+https://  Authorization: Bearer <teleport-jwt>
  ↓
AgentCore Gateway
  │  Validates JWT via Teleport OIDC discovery URL
  ↓
REQUEST Interceptor Lambda
  │  Decodes JWT → extracts sub + roles
  │  Calls Amazon Verified Permissions (Cedar policy)
  │  DENY → MCP error  /  ALLOW → inject identity into request
  ↓
Tool Lambda
  │  whoami_tool     → verified caller identity
  │  get_order_tool  → scoped to caller (Cedar: mcp-user allowed)
  │  update_order_tool → admin-only  (Cedar: aws-personal-admin required)
```

### Tutorial Details

| | |
|:---|:---|
| **Tutorial type** | Interactive |
| **AgentCore components** | AgentCore Gateway |
| **Identity provider** | Teleport (OIDC) |
| **Authorization** | Amazon Verified Permissions (Cedar policies) |
| **Gateway target** | AWS Lambda |
| **MCP transport** | `mcp+https` (Streamable HTTP) |
| **SDK** | boto3 / tsh CLI |

### Prerequisites

- AWS credentials with permissions for Lambda, IAM, bedrock-agentcore, verifiedpermissions
- Teleport cluster with `tsh` installed and `tsh login` completed
- Teleport app service running (`teleport.yaml` in this repo)
- Python 3.9+

## Step 1: Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install -r requirements.txt --quiet

## Step 2: AWS Setup

Set your AWS credentials and region. If running in SageMaker or with an instance profile,
the credential lines can stay commented out.

In [ ]:
import os, json, time, io, zipfile, subprocess
import boto3
from dotenv import load_dotenv
from botocore.exceptions import ClientError

load_dotenv(dotenv_path='.env', override=True)
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')
REGION = os.environ['AWS_DEFAULT_REGION']

# Explicit session forces boto3 to read from os.environ rather than its credential cache
session = boto3.Session(
    aws_access_key_id=os.environ.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.environ.get('AWS_SECRET_ACCESS_KEY'),
    aws_session_token=os.environ.get('AWS_SESSION_TOKEN'),
    region_name=REGION
)

iam            = session.client('iam')
lambda_client  = session.client('lambda')
gateway_client = session.client('bedrock-agentcore-control')
account_id     = session.client('sts').get_caller_identity()['Account']

print(f'Account : {account_id}')
print(f'Region  : {REGION}')

RESOURCE_PREFIX  = os.environ.get('RESOURCE_PREFIX', 'teleport-demo')
TELEPORT_CLUSTER = os.environ.get('TELEPORT_CLUSTER', '')
TARGET_NAME      = 'TeleportDemo'
TAG_CREATOR      = os.environ.get('TAG_CREATOR', '')
TAG_DEMO         = os.environ.get('TAG_DEMO', 'teleport-agentcore')


## Step 3: Create IAM Role for the Tool Lambda

In [ ]:
TOOL_LAMBDA_ROLE_NAME = f'{RESOURCE_PREFIX}-tool-lambda-role'

trust_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Principal': {'Service': 'lambda.amazonaws.com'},
        'Action': 'sts:AssumeRole'
    }]
}

try:
    resp = iam.create_role(
        RoleName=TOOL_LAMBDA_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='Tool Lambda role for Teleport AgentCore demo'
    )
    tool_lambda_role_arn = resp['Role']['Arn']
    print(f'Created role: {tool_lambda_role_arn}')
    time.sleep(10)
except ClientError as e:
    if e.response['Error']['Code'] == 'EntityAlreadyExists':
        tool_lambda_role_arn = iam.get_role(RoleName=TOOL_LAMBDA_ROLE_NAME)['Role']['Arn']
        print(f'Role already exists: {tool_lambda_role_arn}')
    else:
        raise

iam.attach_role_policy(
    RoleName=TOOL_LAMBDA_ROLE_NAME,
    PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
)
print('Attached AWSLambdaBasicExecutionRole')

## Step 4: Package and Deploy the Tool Lambda

The tool Lambda handles three tools:
- **`whoami_tool`** — returns the verified caller identity injected by the interceptor
- **`get_order_tool`** — returns order status scoped to the caller
- **`update_order_tool`** — admin-only mutation (Cedar will gate this)

Identity fields (`_teleport_user`, `_teleport_roles`) are injected by the REQUEST interceptor
Lambda in a later step. Until then they return a placeholder.

In [ ]:
import io, zipfile

with open('lambda_tool.py', 'r') as f:
    lambda_code = f.read()

buf = io.BytesIO()
with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('lambda_function.py', lambda_code)
buf.seek(0)
deployment_package = buf.read()

print(f'Packaged lambda_tool.py ({len(deployment_package):,} bytes)')

In [ ]:
TOOL_LAMBDA_NAME = f'{RESOURCE_PREFIX}-tool-lambda'

tags = {k: v for k, v in {'teleport.dev/creator': TAG_CREATOR, 'demo': TAG_DEMO}.items() if v}

try:
    resp = lambda_client.create_function(
        FunctionName=TOOL_LAMBDA_NAME,
        Runtime='python3.12',
        Role=tool_lambda_role_arn,
        Handler='lambda_function.lambda_handler',
        Code={'ZipFile': deployment_package},
        Description='Identity-aware tool Lambda for Teleport AgentCore demo',
        Timeout=30,
        MemorySize=128,
        **({'Tags': tags} if tags else {}),
    )
    tool_lambda_arn = resp['FunctionArn']
    print(f'Created Lambda: {tool_lambda_arn}')
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceConflictException':
        resp = lambda_client.update_function_code(
            FunctionName=TOOL_LAMBDA_NAME,
            ZipFile=deployment_package
        )
        tool_lambda_arn = resp['FunctionArn']
        print(f'Updated existing Lambda: {tool_lambda_arn}')
    else:
        raise

lambda_client.get_waiter('function_active_v2').wait(FunctionName=TOOL_LAMBDA_NAME)
print('Lambda is active')


## Step 5: Create IAM Role for AgentCore Gateway

The Gateway needs an IAM role to invoke the tool Lambda on outbound calls.

In [ ]:
GATEWAY_ROLE_NAME = f'{RESOURCE_PREFIX}-agentcore-gateway-role'

gateway_trust_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Principal': {'Service': 'bedrock-agentcore.amazonaws.com'},
        'Action': 'sts:AssumeRole',
        'Condition': {
            'StringEquals': {'aws:SourceAccount': account_id},
            'ArnLike': {'aws:SourceArn': f'arn:aws:bedrock-agentcore:{REGION}:{account_id}:*'}
        }
    }]
}

gateway_role_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Action': [
            'lambda:InvokeFunction',
            'bedrock-agentcore:*',
            'secretsmanager:GetSecretValue'
        ],
        'Resource': '*'
    }]
}

try:
    resp = iam.create_role(
        RoleName=GATEWAY_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(gateway_trust_policy),
        Description='AgentCore Gateway role for Teleport demo'
    )
    gateway_role_arn = resp['Role']['Arn']
    print(f'Created gateway role: {gateway_role_arn}')
    time.sleep(10)
except ClientError as e:
    if e.response['Error']['Code'] == 'EntityAlreadyExists':
        gateway_role_arn = iam.get_role(RoleName=GATEWAY_ROLE_NAME)['Role']['Arn']
        print(f'Role already exists: {gateway_role_arn}')
    else:
        raise

iam.put_role_policy(
    RoleName=GATEWAY_ROLE_NAME,
    PolicyName='GatewayPolicy',
    PolicyDocument=json.dumps(gateway_role_policy)
)
print('Policy attached')

## Step 6: Create AgentCore Gateway with Teleport JWT Authorizer

The gateway is configured to validate inbound JWTs against Teleport's OIDC discovery URL.
The `allowedAudiences` must match the `mcp+https://` URI that Teleport puts in the JWT `aud` claim.

In [ ]:
TELEPORT_DISCOVERY = f'https://{TELEPORT_CLUSTER}/.well-known/openid-configuration'
GATEWAY_NAME       = f'{RESOURCE_PREFIX}-identity-demo'

# Check if gateway already exists
existing = [g for g in gateway_client.list_gateways(maxResults=100)['items']
            if g['name'] == GATEWAY_NAME]

if existing:
    gateway_id  = existing[0]['gatewayId']
    gateway_url = gateway_client.get_gateway(gatewayIdentifier=gateway_id)['gatewayUrl']
    print(f'Gateway already exists: {gateway_id}')
else:
    resp = gateway_client.create_gateway(
        name=GATEWAY_NAME,
        roleArn=gateway_role_arn,
        protocolType='MCP',
        authorizerType='CUSTOM_JWT',
        authorizerConfiguration={
            'customJWTAuthorizer': {
                'discoveryUrl': TELEPORT_DISCOVERY,
                'customClaims': [{
                    'inboundTokenClaimName': 'roles',
                    'inboundTokenClaimValueType': 'STRING_ARRAY',
                    'authorizingClaimMatchValue': {
                        'claimMatchValue': {'matchValueString': 'mcp-user'},
                        'claimMatchOperator': 'CONTAINS'
                    }
                }]
            }
        },
        description='Teleport identity demo — JWT validated against Teleport OIDC'
    )
    gateway_id  = resp['gatewayId']
    gateway_url = resp['gatewayUrl']
    print(f'Created gateway: {gateway_id}')

print(f'Gateway URL: {gateway_url}')

In [ ]:
import time

base_url = gateway_url.rstrip('/').removesuffix('/mcp')
mcp_audience = f'mcp+{base_url}/mcp'

# Wait for gateway to leave CREATING state before updating
print('Waiting for gateway to become ready...', end='', flush=True)
for _ in range(60):
    status = gateway_client.get_gateway(gatewayIdentifier=gateway_id).get('status', '')
    if status not in ('CREATING', 'UPDATING'):
        break
    print('.', end='', flush=True)
    time.sleep(5)
else:
    raise TimeoutError(f'Gateway did not become ready in time (last status: {status})')
print(f' {status}')

if status != 'READY':
    raise RuntimeError(f'Gateway is in unexpected state: {status}')

print(f'Setting allowed audience: {mcp_audience}')

gateway_client.update_gateway(
    gatewayIdentifier=gateway_id,
    name=GATEWAY_NAME,
    roleArn=gateway_role_arn,
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration={
        'customJWTAuthorizer': {
            'discoveryUrl': TELEPORT_DISCOVERY,
            'allowedAudience': [mcp_audience],
            'customClaims': [{
                'inboundTokenClaimName': 'roles',
                'inboundTokenClaimValueType': 'STRING_ARRAY',
                'authorizingClaimMatchValue': {
                    'claimMatchValue': {'matchValueString': 'mcp-user'},
                    'claimMatchOperator': 'CONTAINS'
                }
            }]
        }
    }
)
print('Gateway authorizer updated: audience + mcp-user role required')


## Step 7: Register Tool Lambda as Gateway Target

Registers the three tools (`whoami_tool`, `get_order_tool`, `update_order_tool`) as MCP tools
exposed through the gateway.

In [ ]:
# TARGET_NAME is read from .env in the setup cell

tool_schema = {
    'mcp': {
        'lambda': {
            'lambdaArn': tool_lambda_arn,
            'toolSchema': {
                'inlinePayload': [
                    {
                        'name': 'whoami_tool',
                        'description': 'Returns the verified Teleport identity of the caller',
                        'inputSchema': {'type': 'object', 'properties': {}}
                    },
                    {
                        'name': 'get_order_tool',
                        'description': 'Get order status — readable by mcp-user role',
                        'inputSchema': {
                            'type': 'object',
                            'properties': {'orderId': {'type': 'string'}},
                            'required': ['orderId']
                        }
                    },
                    {
                        'name': 'update_order_tool',
                        'description': 'Update order — requires aws-personal-admin role',
                        'inputSchema': {
                            'type': 'object',
                            'properties': {'orderId': {'type': 'string'}},
                            'required': ['orderId']
                        }
                    }
                ]
            }
        }
    }
}

resp = gateway_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name=TARGET_NAME,
    description='Identity-aware tools for Teleport demo',
    targetConfiguration=tool_schema,
    credentialProviderConfigurations=[{'credentialProviderType': 'GATEWAY_IAM_ROLE'}]
)
target_id = resp['targetId']
print(f'Target created: {target_id}')
print(f'Tools: {[t["name"] for t in tool_schema["mcp"]["lambda"]["toolSchema"]["inlinePayload"]]}')

## Step 8: Generate Teleport App Config

Writes `teleport.yaml` from the values already loaded in this notebook — the gateway URL, cluster, and token. `APP_NAME` must match the `name:` field in `app_service.apps` and the `tsh mcp connect` command in Step 9.

In [ ]:
from pathlib import Path

TELEPORT_TOKEN = os.environ.get('TELEPORT_TOKEN', '')
if not TELEPORT_TOKEN:
    raise ValueError('Set TELEPORT_TOKEN in .env  (tctl tokens add --type=app --ttl=1h)')

APP_NAME = 'agentcore-gateway'
data_dir = Path.cwd() / 'teleport-data'
data_dir.mkdir(exist_ok=True)

app_uri = f'mcp+{gateway_url}'

Path('teleport.yaml').write_text(f"""\
version: v3
teleport:
  proxy_server: {TELEPORT_CLUSTER}:443
  auth_token: {TELEPORT_TOKEN}
  data_dir: {data_dir}
  log:
    output: stderr
    severity: INFO
    format:
      output: text
      extra_fields: [timestamp, level, component, caller]

auth_service:
  enabled: false

proxy_service:
  enabled: false

ssh_service:
  enabled: false

app_service:
  enabled: true
  apps:
    - name: {APP_NAME}
      uri: "{app_uri}"
      labels:
        env: dev
      rewrite:
        headers:
          - "Authorization: Bearer {{{{internal.id_token}}}}"
""")

print(f'Written teleport.yaml')
print(f'  proxy_server : {TELEPORT_CLUSTER}:443')
print(f'  auth_token   : {TELEPORT_TOKEN[:8]}…')
print(f'  data_dir     : {data_dir}')
print(f'  app name     : {APP_NAME}')
print(f'  backend      : {app_uri}')
print()
print(f'Start the agent:  teleport start --config teleport.yaml')
print(f'Connect with:     tsh mcp connect {APP_NAME}')


## Step 9: Smoke Test via tsh

Verify the gateway is reachable and tools are listed correctly.
This requires `tsh` to be installed and logged in.

In [ ]:
import subprocess, json as _json

# APP_NAME is set in Step 8 — must match name: in teleport.yaml

msgs = [
    '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2024-11-05","capabilities":{},"clientInfo":{"name":"notebook","version":"0.0.1"}}}',
    '{"jsonrpc":"2.0","id":2,"method":"tools/list","params":{}}',
]

result = subprocess.run(
    ['tsh', 'mcp', 'connect', APP_NAME],
    input='\n'.join(msgs) + '\n',
    capture_output=True, text=True, timeout=30
)

for line in result.stdout.splitlines():
    try:
        print(_json.dumps(_json.loads(line), indent=2))
    except Exception:
        pass

if result.returncode != 0 and result.stderr:
    errors = [l for l in result.stderr.splitlines() if '"level":"error"' in l or 'ERROR' in l]
    for e in errors:
        print('ERROR:', e)


## What's Next

This notebook sets up the foundation. The next notebooks add:

| Notebook | What it adds |
|:---|:---|
| `02-interceptor-identity-injection.ipynb` | REQUEST interceptor Lambda that decodes the Teleport JWT and injects `_teleport_user` + `_teleport_roles` into every tool call |
| `03-cedar-avp-authorization.ipynb` | Amazon Verified Permissions policy store with Cedar policies gating tool access by Teleport role. Live policy change demo (no redeploy needed) |
| `04-strands-agent-demo.ipynb` | Strands agent using the gateway as its tool source, showing the full end-to-end with a natural language prompt |

See `../DEMO-PLAN.md` for the full architecture and Scenario 2 (RFC 8693 Keycloak OBO).

## Cleanup

In [ ]:
# ── Cleanup: safe to run at any point, even in a fresh kernel ──────────────
import os, time
import boto3
from dotenv import load_dotenv
from botocore.exceptions import ClientError

load_dotenv(dotenv_path='.env', override=True)
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')
_s = boto3.Session(
    aws_access_key_id=os.environ.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.environ.get('AWS_SECRET_ACCESS_KEY'),
    aws_session_token=os.environ.get('AWS_SESSION_TOKEN'),
    region_name=os.environ['AWS_DEFAULT_REGION'],
)
_iam    = _s.client('iam')
_lambda = _s.client('lambda')
_gw     = _s.client('bedrock-agentcore-control')

_PREFIX       = os.environ.get('RESOURCE_PREFIX', 'teleport-demo')
_GATEWAY_NAME = f'{_PREFIX}-identity-demo'
_TOOL_LAMBDA  = f'{_PREFIX}-tool-lambda'
_ROLES        = [f'{_PREFIX}-tool-lambda-role', f'{_PREFIX}-agentcore-gateway-role']

def _swallow(fn, *args, **kwargs):
    """Call fn; print and continue on any ClientError."""
    try:
        return fn(*args, **kwargs)
    except ClientError as e:
        print(f'  skip ({e.response["Error"]["Code"]})')
    except Exception as e:
        print(f'  skip ({e})')

# Delete gateway targets + gateway
gw = next((g for g in _gw.list_gateways(maxResults=100).get('items', [])
           if g['name'] == _GATEWAY_NAME), None)
if gw:
    targets = _gw.list_gateway_targets(
        gatewayIdentifier=gw['gatewayId'], maxResults=100).get('items', [])
    for t in targets:
        _swallow(_gw.delete_gateway_target,
                 gatewayIdentifier=gw['gatewayId'], targetId=t['targetId'])
        print(f'Deleted target: {t["name"]}')
    if targets:
        time.sleep(5)
    _swallow(_gw.delete_gateway, gatewayIdentifier=gw['gatewayId'])
    print(f'Deleted gateway: {_GATEWAY_NAME}')
else:
    print(f'Gateway {_GATEWAY_NAME!r} not found — skipping')

# Delete Lambda
_swallow(_lambda.delete_function, FunctionName=_TOOL_LAMBDA)
print(f'Deleted Lambda: {_TOOL_LAMBDA}')

# Delete IAM roles (detach policies first)
for role in _ROLES:
    try:
        for p in _iam.list_attached_role_policies(RoleName=role).get('AttachedPolicies', []):
            _swallow(_iam.detach_role_policy, RoleName=role, PolicyArn=p['PolicyArn'])
        for p in _iam.list_role_policies(RoleName=role).get('PolicyNames', []):
            _swallow(_iam.delete_role_policy, RoleName=role, PolicyName=p)
        _swallow(_iam.delete_role, RoleName=role)
        print(f'Deleted role: {role}')
    except ClientError as e:
        print(f'  Role {role!r}: skip ({e.response["Error"]["Code"]})')

print('Cleanup complete.')
